In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def run_advanced_trade_eda():
    print("=== STARTING ADVANCED TRADE EDA ===")
    
    # 1. Load Data
    price_files = ['prices_round_1_day_-2.csv', 'prices_round_1_day_-1.csv', 'prices_round_1_day_0.csv']
    trade_files = ['trades_round_1_day_-2.csv', 'trades_round_1_day_-1.csv', 'trades_round_1_day_0.csv']
    
    all_prices = []
    all_trades = []
    
    for i, (pf, tf) in enumerate(zip(price_files, trade_files)):
        day = i - 2
        p = pd.read_csv(pf, sep=';')
        t = pd.read_csv(tf, sep=';')
        p['day'] = day
        t['day'] = day
        all_prices.append(p)
        all_trades.append(t)
        
    df_p = pd.concat(all_prices, ignore_index=True)
    df_t = pd.concat(all_trades, ignore_index=True)
    
    # Continuous timestamp for cross-day analysis
    df_p['time_cont'] = (df_p['day'] + 2) * 1_000_000 + df_p['timestamp']
    df_t['time_cont'] = (df_t['day'] + 2) * 1_000_000 + df_t['timestamp']
    
    products = df_t['symbol'].unique()
    
    for prod in products:
        print(f"\n--- Analysis for: {prod} ---")
        dt = df_t[df_t['symbol'] == prod].copy()
        dp = df_p[df_p['product'] == prod].sort_values('time_cont').copy()
        
        # [1] Group trades by lot size (Clustering)
        size_counts = dt['quantity'].value_counts().sort_index()
        print(f"Top 3 Most Frequent Trade Sizes: \n{size_counts.nlargest(3)}")
        
        # [2] Plot when in the day they occur
        plt.figure(figsize=(12, 5))
        plt.scatter(dt['time_cont'], dt['quantity'], alpha=0.4, s=15, c='blue')
        plt.title(f"{prod} - Trade Sizes vs Time")
        plt.xlabel("Continuous Timestamp")
        plt.ylabel("Lot Size")
        plt.savefig(f"{prod}_temporal_sizes.png")
        plt.close()

        # [3] Check sizes at daily High/Low
        for day in [-2, -1, 0]:
            day_p = dp[dp['day'] == day]
            if day_p.empty: continue
            hi_val = day_p['mid_price'].max()
            lo_val = day_p['mid_price'].min()
            
            day_t = dt[dt['day'] == day]
            # Trades at the top/bottom 0.01% of price range
            hi_trades = day_t[day_t['price'] >= hi_val - 2] 
            lo_trades = day_t[day_t['price'] <= lo_val + 2]
            
            if not hi_trades.empty:
                print(f"Day {day} High ({hi_val}): Sizes seen = {hi_trades['quantity'].unique()}")
            if not lo_trades.empty:
                print(f"Day {day} Low ({lo_val}): Sizes seen = {lo_trades['quantity'].unique()}")

        # [4] Direction vs Subsequent Price Moves (Predictive Alpha)
        # Infer direction (Buy=1 if price > mid, Sell=-1 if price < mid)
        dt = pd.merge_asof(dt.sort_values('time_cont'), 
                           dp[['time_cont', 'mid_price']].sort_values('time_cont'), 
                           on='time_cont', direction='backward')
        dt['direction'] = np.where(dt['price'] > dt['mid_price'], 1, -1)
        
        # Calculate 10-step forward return
        dp['fwd_ret_10'] = dp['mid_price'].shift(-10) / dp['mid_price'] - 1
        dt = pd.merge_asof(dt.sort_values('time_cont'), 
                           dp[['time_cont', 'fwd_ret_10']], 
                           on='time_cont', direction='forward')
        
        # Group by size and direction to see who has the most "impact"
        impact = dt.groupby(['quantity', 'direction'])['fwd_ret_10'].mean().unstack()
        print("\nPredictive Alpha (Avg 10-tick Return after trade):")
        print(impact)
        
        # Plotting the impact for the most common sizes
        impact.plot(kind='bar', figsize=(10, 6))
        plt.title(f"{prod} - Predictive Impact by Size & Direction")
        plt.axhline(0, color='black', lw=1)
        plt.savefig(f"{prod}_trade_impact.png")
        plt.close()

if __name__ == "__main__":
    run_advanced_trade_eda()

=== STARTING ADVANCED TRADE EDA ===

--- Analysis for: ASH_COATED_OSMIUM ---
Top 3 Most Frequent Trade Sizes: 
quantity
6    230
5    224
3    179
Name: count, dtype: int64
Day -2 High (10019.0): Sizes seen = [3 4 8 9 6]
Day -1 High (10019.0): Sizes seen = [ 3 10  2  6]
Day 0 High (10023.0): Sizes seen = [6 2 9]

Predictive Alpha (Avg 10-tick Return after trade):
direction        -1         1
quantity                     
2         -0.000051 -0.000006
3         -0.000004 -0.000033
4         -0.000064  0.000005
5         -0.000011 -0.007796
6          0.000006 -0.000013
7         -0.000025 -0.000035
8          0.000013 -0.000037
9          0.000086 -0.000087
10         0.000005 -0.000011

--- Analysis for: INTARIAN_PEPPER_ROOT ---
Top 3 Most Frequent Trade Sizes: 
quantity
6    219
3    198
5    195
Name: count, dtype: int64
Day -1 High (12006.0): Sizes seen = [6]
Day 0 High (13007.0): Sizes seen = [4]

Predictive Alpha (Avg 10-tick Return after trade):
direction        -1         1
qua

In [2]:
import pandas as pd
import numpy as np

def analyze_liquidity_taking():
    # Load Data
    price_files = ['prices_round_1_day_-2.csv', 'prices_round_1_day_-1.csv', 'prices_round_1_day_0.csv']
    all_prices = []
    for i, f in enumerate(price_files):
        df = pd.read_csv(f, sep=';')
        df['day'] = i - 2
        all_prices.append(df)
    df_p = pd.concat(all_prices, ignore_index=True)

    products = ["ASH_COATED_OSMIUM", "INTARIAN_PEPPER_ROOT"]

    for prod in products:
        print(f"\n=== Analyzing {prod} ===")
        df = df_p[df_p['product'] == prod].sort_values(['day', 'timestamp']).copy()
        
        # Define Fair Value (Matches our Champion Bot)
        if prod == "ASH_COATED_OSMIUM":
            df['fair_value'] = 10000
        else:
            # VAMP for Pepper Root
            df['vamp'] = (df['bid_price_1'] * df['ask_volume_1'] + df['ask_price_1'] * df['bid_volume_1']) / (df['bid_volume_1'] + df['ask_volume_1'])
            df['fair_value'] = df['vamp'].rolling(window=15).mean()
        
        df = df.dropna(subset=['fair_value', 'ask_price_1', 'bid_price_1'])
        
        # Calculate 'Take' Opportunities
        # Buy when market Ask is cheaper than our Fair Value
        df['buy_take'] = df['ask_price_1'] < df['fair_value']
        df['buy_edge'] = np.where(df['buy_take'], df['fair_value'] - df['ask_price_1'], 0)
        
        # Sell when market Bid is higher than our Fair Value
        df['sell_take'] = df['bid_price_1'] > df['fair_value']
        df['sell_edge'] = np.where(df['sell_take'], df['bid_price_1'] - df['fair_value'], 0)
        
        # Reporting
        total = len(df)
        b_count = df['buy_take'].sum()
        s_count = df['sell_take'].sum()
        
        print(f"Total Ticks Analyzed: {total}")
        print(f"Opportunities to 'Take' Ask (Buy): {b_count} ({b_count/total:.2%})")
        print(f"Average Edge on Buy: {df[df['buy_take']]['buy_edge'].mean():.2f} ticks")
        print(f"Opportunities to 'Take' Bid (Sell): {s_count} ({s_count/total:.2%})")
        print(f"Average Edge on Sell: {df[df['sell_take']]['sell_edge'].mean():.2f} ticks")

if __name__ == "__main__":
    analyze_liquidity_taking()


=== Analyzing ASH_COATED_OSMIUM ===
Total Ticks Analyzed: 27644
Opportunities to 'Take' Ask (Buy): 1377 (4.98%)
Average Edge on Buy: 2.77 ticks
Opportunities to 'Take' Bid (Sell): 1291 (4.67%)
Average Edge on Sell: 3.03 ticks

=== Analyzing INTARIAN_PEPPER_ROOT ===
Total Ticks Analyzed: 8989
Opportunities to 'Take' Ask (Buy): 142 (1.58%)
Average Edge on Buy: 2.86 ticks
Opportunities to 'Take' Bid (Sell): 141 (1.57%)
Average Edge on Sell: 3.39 ticks


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def run_trader_id_eda():
    # 1. Load and Combine Trade Data
    trade_files = ['trades_round_1_day_-2.csv', 'trades_round_1_day_-1.csv', 'trades_round_1_day_0.csv']
    all_trades = []
    
    for i, tf in enumerate(trade_files):
        day = i - 2
        try:
            t = pd.read_csv(tf, sep=';')
            t['day'] = day
            all_trades.append(t)
        except Exception as e:
            print(f"Skipping {tf}: {e}")
            
    if not all_trades:
        print("No trade data found.")
        return

    df_t = pd.concat(all_trades, ignore_index=True)
    
    # Check for IDs
    if df_t['buyer'].isna().all() and df_t['seller'].isna().all():
        print("ALERT: Trader IDs (buyer/seller) are empty in this dataset. Plotting will be skipped.")
        return

    # Restructure data: create an 'activity' log where every row is a trader's action
    # (One trade row creates one 'BUY' action and one 'SELL' action)
    buyers = df_t[['timestamp', 'buyer', 'symbol', 'price', 'quantity', 'day']].copy()
    buyers.columns = ['timestamp', 'trader', 'symbol', 'price', 'quantity', 'day']
    buyers['side'] = 'BUY'
    
    sellers = df_t[['timestamp', 'seller', 'symbol', 'price', 'quantity', 'day']].copy()
    sellers.columns = ['timestamp', 'trader', 'symbol', 'price', 'quantity', 'day']
    sellers['side'] = 'SELL'
    
    activity = pd.concat([buyers, sellers], ignore_index=True).dropna(subset=['trader'])
    activity['time_cont'] = (activity['day'] + 2) * 1_000_000 + activity['timestamp']
    
    # 1. Group all trades by counterparty - Identify the most active ones
    top_traders = activity['trader'].value_counts().nlargest(5).index
    print(f"Top 5 most active traders identified: {list(top_traders)}")
    
    # Filter for top traders to keep plots readable
    top_activity = activity[activity['trader'].isin(top_traders)]

    # 2. Plot trade size distribution per counterparty
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=top_activity, x='trader', y='quantity', hue='side', palette="Set2")
    plt.title("Trade Size Distribution per Top Counterparty (Buy vs Sell)")
    plt.grid(axis='y', alpha=0.3)
    plt.savefig("trader_size_distribution.png", dpi=300)
    print("Saved: trader_size_distribution.png")
    
    # 3. Plot time-of-day patterns (Activity bursts)
    plt.figure(figsize=(14, 6))
    for trader in top_traders:
        trader_data = activity[activity['trader'] == trader]
        plt.scatter(trader_data['time_cont'], [trader]*len(trader_data), alpha=0.6, s=20, label=f"Trader {trader}")
    plt.title("Trader Activity Timelines (Day -2 to Day 0)")
    plt.xlabel("Continuous Timestamp")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig("trader_time_patterns.png", dpi=300)
    print("Saved: trader_time_patterns.png")
    
    # 4. Plot direction (buy/sell) vs. price level
    # We Z-score the price per symbol to see if traders buy 'low' and sell 'high' relative to the mean
    activity['price_z'] = activity.groupby('symbol')['price'].transform(lambda x: (x - x.mean()) / x.std())
    
    plt.figure(figsize=(12, 7))
    sns.violinplot(data=top_activity, x='trader', y='price_z', hue='side', split=True, inner="quart", palette="muted")
    plt.axhline(0, color='red', linestyle='--', alpha=0.5)
    plt.title("Trader Direction vs. Normalized Price Level (Z-Score)")
    plt.ylabel("Price Level (Std Dev from Mean)")
    plt.savefig("trader_direction_vs_price.png", dpi=300)
    print("Saved: trader_direction_vs_price.png")

if __name__ == "__main__":
    run_trader_id_eda()

ALERT: Trader IDs (buyer/seller) are empty in this dataset. Plotting will be skipped.


In [4]:
import pandas as pd
import numpy as np

def verify_8_lot_signal():
    print("=== Verifying 8-Lot Momentum Signal on Pepper Root ===")
    
    price_files = ['prices_round_1_day_-2.csv', 'prices_round_1_day_-1.csv', 'prices_round_1_day_0.csv']
    trade_files = ['trades_round_1_day_-2.csv', 'trades_round_1_day_-1.csv', 'trades_round_1_day_0.csv']
    
    all_prices = []
    all_trades = []
    
    for i, (pf, tf) in enumerate(zip(price_files, trade_files)):
        day = i - 2
        p = pd.read_csv(pf, sep=';')
        t = pd.read_csv(tf, sep=';')
        p['day'] = day
        t['day'] = day
        all_prices.append(p)
        all_trades.append(t)
        
    df_p = pd.concat(all_prices, ignore_index=True)
    df_t = pd.concat(all_trades, ignore_index=True)
    
    df_p['time_cont'] = (df_p['day'] + 2) * 1_000_000 + df_p['timestamp']
    df_t['time_cont'] = (df_t['day'] + 2) * 1_000_000 + df_t['timestamp']
    
    # Filter for Pepper Root
    dt = df_t[df_t['symbol'] == 'INTARIAN_PEPPER_ROOT'].copy()
    dp = df_p[df_p['product'] == 'INTARIAN_PEPPER_ROOT'].sort_values('time_cont').copy()
    
    # Match trades to closest previous mid_price to find aggressive direction
    dt = pd.merge_asof(dt.sort_values('time_cont'), 
                       dp[['time_cont', 'mid_price']].sort_values('time_cont'), 
                       on='time_cont', direction='backward')
    
    dt['direction'] = np.where(dt['price'] > dt['mid_price'], 1, -1)
    
    # Calculate future mid_price 1 tick (100ms) and 2 ticks (200ms) ahead
    dp['fwd_mid_1'] = dp['mid_price'].shift(-1)
    dp['fwd_mid_2'] = dp['mid_price'].shift(-2)
    
    # Merge future prices back to trades
    dt = pd.merge_asof(dt.sort_values('time_cont'), 
                       dp[['time_cont', 'fwd_mid_1', 'fwd_mid_2']], 
                       on='time_cont', direction='forward')
    
    # Check if the trade direction matched the future price movement
    dt['win_1_tick'] = ((dt['fwd_mid_1'] - dt['mid_price']) * dt['direction']) > 0
    dt['win_2_ticks'] = ((dt['fwd_mid_2'] - dt['mid_price']) * dt['direction']) > 0
    
    # Filter only 8-lot trades
    eight_lots = dt[dt['quantity'] == 8]
    
    for day in [-2, -1, 0]:
        day_data = eight_lots[eight_lots['day'] == day]
        if not day_data.empty:
            win_1 = day_data['win_1_tick'].mean() * 100
            win_2 = day_data['win_2_ticks'].mean() * 100
            count = len(day_data)
            print(f"Day {day}: Found {count} 8-lot trades. Win Rate (+1 tick): {win_1:.1f}%. Win Rate (+2 ticks): {win_2:.1f}%")

if __name__ == "__main__":
    verify_8_lot_signal()

=== Verifying 8-Lot Momentum Signal on Pepper Root ===
Day -2: Found 10 8-lot trades. Win Rate (+1 tick): 80.0%. Win Rate (+2 ticks): 80.0%
Day -1: Found 12 8-lot trades. Win Rate (+1 tick): 83.3%. Win Rate (+2 ticks): 83.3%
Day 0: Found 20 8-lot trades. Win Rate (+1 tick): 90.0%. Win Rate (+2 ticks): 95.0%
